# Module 3: Parallel / Fork-Join (15 min)

Apply **Pattern 2**: fork independent sub-tasks to run simultaneously, then merge results — latency drops to the slowest single worker.

```
Decision Brief
      │
      ▼
 ┌────────────┐  (sequential)
 │ Researcher │  gathers shared context
 └──────┬─────┘
        │ research_text (shared)
   ┌────┴────┬────────┐
   ▼         ▼        ▼    asyncio.gather → all 3 run at once
 Analyzer A  B        C
   └────┬────┴────────┘
        │ join: wait for all 3
        ▼
 ┌─────────────┐  (sequential)
 │ Synthesizer │  merges all analyses
 └─────────────┘
```

**When to use this pattern:**
- Sub-tasks are independent of each other
- Latency / throughput matters
- Results can be merged deterministically

**Key Strands API:** `agent.invoke_async(prompt)` + `asyncio.gather(...)`

**Prerequisites:** Complete Modules 1 and 2 — this module reuses Module 1 tools.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns NovaCart financial/operational data | CLV, churn rate, revenue, top-spender metrics |
| `get_market_benchmarks(industry)` | Returns e-commerce benchmarks | Avg CLV, churn, subscription rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns competitor premium tier data | Pricing, pilot approach, adoption %, CLV lift, payback period |

> **Note:** Only the **Researcher** has these tools. The three Analyzers run with no tools — they receive the research context in their input prompt. This is intentional: tool access is granted per-agent based on role.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1 — Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2 — Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3 — Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4 — Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")

---

## Part 1 — Setup and Prompts

`nest_asyncio.apply()` patches Jupyter's event loop so `asyncio.gather` works inside a notebook cell. In a regular Python script (`chat.py`), you use `asyncio.run()` instead.

In [ ]:
import sys, os, asyncio, time
sys.path.insert(0, os.path.join(os.getcwd(), "..", "01-strands-foundations"))

# nest_asyncio patches Jupyter's event loop so asyncio.gather works in a notebook
import nest_asyncio
nest_asyncio.apply()

from strands import Agent
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

In [ ]:
# ── System prompts ────────────────────────────────────────────────────────

RESEARCHER_PROMPT = '''You are a market research specialist.
Gather company data, benchmarks, and competitor intelligence using your tools.
Return structured findings — data only, no recommendations.'''

# Note: all three analyzers share the same prompt — each receives a DIFFERENT option.
# The option description in the user message is what specializes each one.
ANALYZER_PROMPT = '''You are a business strategy analyst focused on ONE option.
Evaluate the option assigned to you:
- Strengths and weaknesses
- Implementation complexity: Low / Medium / High (one-sentence justification)
- Top 2 risks with specific mitigations
- Verdict: Proceed / Proceed with caution / Do not proceed
Be concise — 150 words max.'''

SYNTHESIZER_PROMPT = '''You are an executive communications specialist.
Given research findings and analyses of three options, write a leadership memo:

## Decision Memo
**Recommendation**: [one sentence — which option and why]

### Options at a Glance
| | Option A | Option B | Option C |
|---|---|---|---|
| Complexity | | | |
| Risk level | | | |
| Verdict | | | |

### Top 3 Risks & Mitigations
### Success Metrics (3-5 KPIs with targets)
### Decision Required: owner · deadline · approvers needed

Under 400 words. Be direct.'''

---

## Part 2 — Research Phase (Sequential)

The Researcher runs first. Its output becomes the **shared context** passed to all three analyzers — so they all start with the same market intelligence.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Options:
  Option A — Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B — Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C — Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# ── Step 1: Research (sequential) ────────────────────────────────────────
# The Researcher runs first to gather shared context.
# All three analyzers will receive the same research findings.
researcher = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=RESEARCHER_PROMPT,
    callback_handler=None,
)

print("Step 1/3 — Researcher gathering shared context...")
t0 = time.time()
research_result = researcher(f"Gather market data for this decision:\n{DECISION_BRIEF}")
research_text = str(research_result)
print(f"  Done in {time.time()-t0:.1f}s | {len(research_text)} chars of research findings")

---

## Part 3 — Fork: 3 Analyzers in Parallel

`invoke_async` returns a coroutine — the agent runs asynchronously. `asyncio.gather` starts all three at once and waits until **all** finish. This is the **fork**. The **join** happens when `gather` returns.

In [ ]:
# ── Step 2: Fork — 3 analyzers in parallel ───────────────────────────────
# Each analyzer gets a DIFFERENT option but the SAME research context.
# invoke_async returns a coroutine. asyncio.gather runs all three simultaneously.

analyzer_a = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
analyzer_b = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
analyzer_c = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)

async def fork():
    return await asyncio.gather(
        analyzer_a.invoke_async(
            f"Analyze: Option A (Exclusive Premium, $19.99/mo, invite-only top 10%)"
            f"\nBrief: {DECISION_BRIEF}\nResearch: {research_text}"
        ),
        analyzer_b.invoke_async(
            f"Analyze: Option B (Gradual Rollout, $14.99/mo, 5% pilot with kill-switch)"
            f"\nBrief: {DECISION_BRIEF}\nResearch: {research_text}"
        ),
        analyzer_c.invoke_async(
            f"Analyze: Option C (Full Launch, $12.99/mo, open to all, 30-day trial)"
            f"\nBrief: {DECISION_BRIEF}\nResearch: {research_text}"
        ),
    )

print("Step 2/3 — Forking: 3 analyzers running simultaneously...")
t1 = time.time()
result_a, result_b, result_c = asyncio.run(fork())
fork_time = time.time() - t1

print(f"  Join complete in {fork_time:.1f}s — all 3 analyzers finished")
print(f"  Sequential equivalent would take ~{fork_time * 3:.0f}s")

In [ ]:
# Inspect what each analyzer produced
print("=== Option A analysis (first 200 chars) ===")
print(str(result_a)[:200], "...")
print()
print("=== Option B analysis (first 200 chars) ===")
print(str(result_b)[:200], "...")
print()
print("=== Option C analysis (first 200 chars) ===")
print(str(result_c)[:200], "...")
print()
print("Each analyzer saw: brief + research + its specific option.")
print("None of them saw what the others were analyzing. That is the fork.")

---

## Part 4 — Synthesize (Sequential)

The Synthesizer runs **after** the join — it needs all three analyses. It receives the brief, the research, and each option's analysis as context.

In [ ]:
# ── Step 3: Synthesize (sequential) ──────────────────────────────────────
# The Synthesizer merges all three analyses. It runs AFTER the join is complete.

synthesizer = Agent(system_prompt=SYNTHESIZER_PROMPT)

print("Step 3/3 — Synthesizer writing the executive memo:")
print("─" * 60)

t2 = time.time()
memo_result = synthesizer(
    f"Brief:\n{DECISION_BRIEF}\n\n"
    f"Research findings:\n{research_text}\n\n"
    f"Option A analysis:\n{result_a}\n\n"
    f"Option B analysis:\n{result_b}\n\n"
    f"Option C analysis:\n{result_c}"
)

print("─" * 60)
print(f"Done in {time.time()-t2:.1f}s")

---

## Part 5 — Pipeline Metrics

In [ ]:
# ── Pipeline metrics ─────────────────────────────────────────────────────
r_s = research_result.metrics.get_summary()
a_s = result_a.metrics.get_summary()
b_s = result_b.metrics.get_summary()
c_s = result_c.metrics.get_summary()
m_s = memo_result.metrics.get_summary()

print("Pipeline metrics:")
print(f"  Researcher:  cycles={r_s.get('total_cycles')} | context_msgs={len(researcher.messages)}")
print(f"  Analyzer A:  cycles={a_s.get('total_cycles')} | context_msgs={len(analyzer_a.messages)}")
print(f"  Analyzer B:  cycles={b_s.get('total_cycles')} | context_msgs={len(analyzer_b.messages)}")
print(f"  Analyzer C:  cycles={c_s.get('total_cycles')} | context_msgs={len(analyzer_c.messages)}")
print(f"  Synthesizer: cycles={m_s.get('total_cycles')} | context_msgs={len(synthesizer.messages)}")
print()
print(f"Fork wall-clock time: {fork_time:.1f}s")
print(f"Sequential equivalent: ~{fork_time * 3:.0f}s")
print(f"Latency reduction: ~{(1 - 1/3)*100:.0f}% (limited by slowest analyzer)")

In [ ]:
# ── Token usage across the pipeline ─────────────────────────────────────────
r_usage = research_result.metrics.get_summary().get("accumulated_usage", {})
a_usage = result_a.metrics.get_summary().get("accumulated_usage", {})
b_usage = result_b.metrics.get_summary().get("accumulated_usage", {})
c_usage = result_c.metrics.get_summary().get("accumulated_usage", {})
m_usage = memo_result.metrics.get_summary().get("accumulated_usage", {})

print(f"{'Stage':<14} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 42)
for label, u in [
    ("Researcher", r_usage),
    ("Analyzer A", a_usage),
    ("Analyzer B", b_usage),
    ("Analyzer C", c_usage),
    ("Synthesizer", m_usage),
]:
    print(f"{label:<14} {u.get('inputTokens', 0):>8} {u.get('outputTokens', 0):>8} {u.get('totalTokens', 0):>8}")
print()
print(f"Fork savings: 3 analyzers ran in {fork_time:.1f}s instead of ~{fork_time*3:.0f}s sequentially")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `invoke_async` | Runs an agent as a coroutine — non-blocking |
| `asyncio.gather` | Fork: starts N coroutines simultaneously, join: waits for all |
| Shared context | All analyzers receive the same research findings |
| Latency benefit | Wall-clock ≈ slowest single analyzer, not sum of all three |

---

## What's Next

In **Module 4: Critic-Refiner**, you add a quality gate — the Critic evaluates the memo and sends feedback if it doesn't meet the bar. The Synthesizer retries until the Critic approves. That is a cycle — Pattern 3 uses `GraphBuilder`.

---

## Want a real multi-turn conversation?

```bash
cd samples/03-parallel-fork-join
pip install -r requirements.txt
python chat.py
```

---

> 💡 **Note — Memory & Observability (review):**
>
> **Memory:** `research_text` is shared with all 3 analyzers by passing it explicitly in each prompt. This is correct for the workshop but could be replaced by `invocation_state` — a shared dict Strands passes to all agents in a pipeline. Review: does `invoke_async` support `invocation_state`? If so, the shared research context could be cleaner. Module 6 (Memory) explores cross-agent state with AgentCore Memory.
>
> **Observability:** The 3 parallel `invoke_async` calls have no visibility — you only see the final memo. Review question: should Module 7 add OTEL spans per `invoke_async` call so you can see A/B/C timing independently in CloudWatch?